# **Part 1:** Data preparation

In [1]:
import html
import json
import re
import nltk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import RegexpTokenizer

nltk.download('stopwords')

# Configure stop words and stemmer
stop_words = set(stopwords.words('english'))
important_words = {'men', 'women', 'no', 'not', 'nor'}
stop_words -= important_words
stemmer = PorterStemmer()

def flatten_product_details(details):
    if not isinstance(details, list):
        return ''
    attributes = []
    for item in details:
        if isinstance(item, dict):
            for key, value in item.items():
                attributes.append(f'{key} {value}')
    return ' '.join(attributes)

def preprocess_text(text):
    if not isinstance(text, str) or not text.strip():
        return []
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.lower()
    tokenizer = RegexpTokenizer(r'\b[a-z0-9]+\b')
    tokens = tokenizer.tokenize(text)
    filtered_tokens = [t for t in tokens if t not in stop_words]
    return [stemmer.stem(t) for t in filtered_tokens]

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\fronk\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
data_path = '../../data/fashion_products_dataset.json'
with open(data_path) as fp:
    data = json.load(fp)

df = pd.DataFrame(data).copy()

# Convert prices, discounts, and ratings to numeric
for col in ['selling_price', 'actual_price']:
    df[col + '_num'] = pd.to_numeric(df[col].astype(str).str.replace(',', '', regex=False), errors='coerce')

df['discount_num'] = pd.to_numeric(df['discount'].astype(str).str.extract(r'(\d+(?:\.\d+)?)')[0], errors='coerce')
df['average_rating_num'] = pd.to_numeric(df['average_rating'].astype(str), errors='coerce')
df['out_of_stock_bool'] = df['out_of_stock'].astype(bool)
df['product_details_text'] = df['product_details'].apply(flatten_product_details)

In [3]:
# Apply preprocessing to all text and categorical fields
other_fields = ['brand', 'category', 'sub_category', 'seller', 'product_details_text']
df['title_processed'] = df['title'].apply(preprocess_text)
df['description_processed'] = df['description'].apply(preprocess_text)

for field in other_fields:
    target_col = 'product_details_processed' if field == 'product_details_text' else f'{field}_processed'
    df[target_col] = df[field].apply(preprocess_text)

# Build structured stores for searching and filtering
separate_fields_corpus = {
    row['pid']: {
        'title': row['title_processed'],
        'description': row['description_processed'],
        'brand': row['brand_processed'],
        'category': row['category_processed'],
        'sub_category': row['sub_category_processed'],
        'seller': row['seller_processed'],
        'product_details': row['product_details_processed'],
    }
    for _, row in df.iterrows()
}

metadata_store = {
    row['pid']: {
        'out_of_stock': row['out_of_stock_bool'],
        'selling_price': row['selling_price_num'],
        'actual_price': row['actual_price_num'],
        'discount_percent': row['discount_num'],
        'average_rating': row['average_rating_num'],
    }
    for _, row in df.iterrows()
}

print(f'Processed {len(df)} products successfully.')

Processed 28080 products successfully.


# **Part 2:** Indexing and evaluation

In [4]:
from collections import defaultdict, Counter
import math
import os

In [5]:
SEARCHING_FIELDS = {
    "title": "title_processed",
    "description": "description_processed",
    "brand": "brand_processed",
    "category": "category_processed",
    "sub_category": "sub_category_processed",
    "product_details": "product_details_processed",
    "seller": "seller_processed"
}

FIELDS_WEIGHTS = {
    "title": 3.0,
    "description": 1.5,
    "brand": 2.0,
    "category": 1.0,
    "sub_category": 1.5,
    "product_details": 1.0,
    "seller": 0.5
}

# Keep only rows with a valid PID
index_df = df.dropna(subset=["pid"]).copy()
index_df["pid"] = index_df["pid"].astype(str)

if index_df["pid"].duplicated().any():
    raise ValueError("PID values must be unique before building the index.")

# Missing processed fields behave as empty token lists
for field, column in SEARCHING_FIELDS.items():
    value = index_df[column]
    if isinstance(value, list):
        index_df[column] = value
    else: []

N_DOCUMENTS = len(index_df)
print("Documents to index:", N_DOCUMENTS)
print("Searchable fields:", list(SEARCHING_FIELDS))

Documents to index: 28080
Searchable fields: ['title', 'description', 'brand', 'category', 'sub_category', 'product_details', 'seller']


In [6]:
inverted_index = {
    field: defaultdict(dict) for field in SEARCHING_FIELDS
}

# Build the inverted index
for _, row in index_df.iterrows():
    pid = row["pid"]

    for field, column in SEARCHING_FIELDS.items():
        tokens = row[column]
        if not isinstance(tokens, list):
            continue
        term_counts = Counter(tokens)

        for term, frequency in term_counts.items():
            inverted_index[field][term][pid] = frequency

# Convert defaultdicts to regular dicts for clear display
inverted_index = {
    field: dict(term_postings) for field, term_postings in inverted_index.items()
}

# Compute the document frequency for each term and the total vocabulary across all fields
document_frequency = defaultdict(int)

all_terms = set()

for field in SEARCHING_FIELDS:
    all_terms.update(inverted_index[field].keys())

for term in all_terms:
    matching_pid = set()
    for field in SEARCHING_FIELDS:
        matching_pid.update(inverted_index[field].get(term, {}).keys())
    document_frequency[term] = len(matching_pid)

vocabulary = set(all_terms)

print("Number of indexed documents:", N_DOCUMENTS)
print("Vocabulary size across searchable fields:", len(vocabulary))
print("Number of terms in each field:")
for field in SEARCHING_FIELDS:
    print(f"  {field}: {len(inverted_index[field])}")

Number of indexed documents: 28080
Vocabulary size across searchable fields: 20031
Number of terms in each field:
  title: 729
  description: 4699
  brand: 402
  category: 7
  sub_category: 39
  product_details: 16770
  seller: 683


In [7]:
def get_postings(term):
    """Returns {pid: {field: term_frequency}} for a term across all fields"""
    result = defaultdict(dict)

    for field in SEARCHING_FIELDS:
        postings = inverted_index[field].get(term, {})
        for pid, frequency in postings.items():
            result[pid][field] = frequency

    return dict(result)

def conjunctive_search(query):
    """Return PIDs containing every preprocessed query term"""
    query_terms = list(dict.fromkeys(preprocess_text(query)))  # Remove duplicates while preserving order

    if not query_terms:
        return []

    term_postings = [get_postings(term) for term in query_terms]

    # If any term has no postings, return an empty list immediately
    if any(not postings for postings in term_postings):
        return []

    # Find the intersection of PIDs across all query terms
    matching_pids = set(term_postings[0].keys())
    for postings in term_postings[1:]:
        matching_pids.intersection_update(postings.keys())

    return list(matching_pids)

# Check the two predefined validation queries from Part 1
validation_queries = [
    "women full sleeve sweatshirt cotton",
    "men slim jeans blue",
]

for query in validation_queries:
    matching_pids = conjunctive_search(query)
    print(f"Query: '{query}'")
    print(f"Matching PIDs: {matching_pids}")
    print(f"Number of matching products: {len(matching_pids)}\n")

Query: 'women full sleeve sweatshirt cotton'
Matching PIDs: ['SWSFW6KB8GZHEC72', 'SWSF9W528G7VEGCV', 'SWSF5R7F2ZYCZYKH', 'SWSFWHEPSKFVHCUZ', 'SWSFZ2H54ZGQTCME', 'SWSFN8XDZF2AWKQT', 'SWSFUNF6XHCGGVCU', 'SWSFN8YZ33MHKQVF', 'SWSFWCXHFRX2DBSG', 'SWSFUFG6BHP5DHB9', 'SWSFQM6ZZQKXFZYM', 'SWSFWCS9DZGKEGVG', 'SWSFQHFRBGTF4ZAM', 'SWSFZ7JXDTXAAZSH', 'SWSFYFFYFXCMPBDZ', 'TSHFUMFC7CTTFP9A', 'SWSEXAHDQQCKCAEA', 'SWSFWEF2NMHRHDUB', 'SWSFRAKJY6ZYPFFU', 'SWSFJY5ZFHQ7HXKW', 'SWSFY4HPTTBCZZAJ', 'SWSFRR7PTQVEEGTR', 'SWSEDQEACHUWZJRK', 'TSHFY24T7PBC7PZR', 'SWSFV5JN8QQYFN4Q', 'SWSFVZRFTM4P9XGF', 'SWSFWG9B7CCYYVZZ', 'SWSFFVKBYG9ZUBTW', 'SWSFUNF6VFJBYZZC', 'SWSFSFEFS5PYTYBF', 'SWSE8FYZQCRNM2KY', 'SWSFWAV3SPSZ36Y4', 'TSHFY9QYNH2N4HYB', 'SWSFXZWHCDGSHDUP', 'SWSFWCSBYWNK2FYY', 'SWSFXMFPDVRHYYPH', 'SWSFPTFJZYHMPPFC', 'SWSEYQR8V9YBQVPD', 'TSHFVGBUGYUHMWGR', 'SWSFWG9BKWPB9U2D', 'SWSFWEYZC9EEYB6E', 'TSHFUMFCK4BMCFZQ', 'SWSFNMS5R9ZQDGPZ', 'SWSF5R7FFZPUHDU5', 'SWSFY36HVGCV8ZKW', 'SWSFUY89SZUZQGFF', 'SWSEXR9GTYZGTRFH',

In [8]:
test_queries = [
    "women hoodie polycotton red graphic print",
    "men wide blue v-neck t-shirt",
    "summer dress pink",
    "yellow running shoes",
    "raincoat waterproof women",
]

for query in test_queries:
    matching_pids = conjunctive_search(query)
    print(f"Query: '{query}'")
    print(f"Matching PIDs: {matching_pids}")
    print(f"Number of matching products: {len(matching_pids)}\n")

Query: 'women hoodie polycotton red graphic print'
Matching PIDs: ['SWSFWCXMJRZDWKVU', 'SWSFWCSNPENAZRZM', 'SWSFWCSPHDFH5YKT', 'SWSFWEYZZZEJEEJW', 'SWSFWEYZVYXHPGQJ', 'SWSFWCSNXA6U57G5', 'SWSFWCXHFRX2DBSG', 'SWSFWEYZC9EEYB6E', 'SWSFWCSPEGMDH8FV', 'SWSFWCSNPEDVPS7Z', 'SWSFWEF36Z3RKTJ7', 'SWSFWCTDHRABJFCR', 'SWSFWCSZGNU7N4F6']
Number of matching products: 13

Query: 'men wide blue v-neck t-shirt'
Matching PIDs: ['TSHEXZ8KTZVQJECU', 'TSHF4ZFXV3VHZ5GU', 'TSHEXZ8KJC9GHZCH', 'TSHFACN2M6QJZG6Z', 'TSHE7GY3D5WWFKZR', 'TSHFNUHHSKB9Y9HM', 'TSHFNUGTDBPCWZ3Q', 'TSHE8UHZ6WFVCCNU', 'TSHFACNYDBQQYHPZ', 'TSHENWW53TW4EGWG', 'TSHFGH2XTNZFHENV']
Number of matching products: 11

Query: 'summer dress pink'
Matching PIDs: ['TSHFHBNTJCCGDG6F', 'TSHFHHRKK9F9MRCW', 'TSHFHBNTYFCCHZTU', 'TSHFHHRKZTMCUUGK', 'TSHFHBNTS6H94M33', 'TSHFHHRKTGVPEZYT', 'TSHFHBNTSG3UQWTV', 'TSHFHBNTNH7XJVH7', 'TSHFHHRKNTCZ9QEW', 'TSHFHBNT8TVE7ZHZ', 'TSHFHBNTFZ589XYW', 'TSHFHBNT4HAKCDBF', 'TSHFHBNTHZB6Z6DN', 'TSHFHHRK5ZBGPHCG', 'TSHFHHRHW

In [9]:
def compute_tfidf_score(pid, query_terms):
    """Calculate a field-weighted TF-IDF score for one document"""
    score = 0.0

    for term in query_terms:
        df_term = document_frequency.get(term, 0)
        if df_term == 0:
            continue  # Skip terms not in the vocabulary

        idf = math.log((N_DOCUMENTS + 1)/(df_term + 1)) + 1  # Smoothed IDF

        for field in SEARCHING_FIELDS:
            tf = inverted_index[field].get(term, {}).get(pid, 0)
            if tf > 0:
                tf_weighted = 1 + math.log(tf)  # Logarithmic TF
                score += tf_weighted * idf * FIELDS_WEIGHTS[field]

    return score

def search_rank(query, topk = 10):
    """Return top-k PIDs ranked by field-weighted TF-IDF score"""
    query_terms = list(dict.fromkeys(preprocess_text(query)))  # Remove duplicates while preserving order

    if not query_terms:
        return []

    candidate_pids = conjunctive_search(query)

    if not candidate_pids:
        return []

    ranked = [(pid, compute_tfidf_score(pid, query_terms)) for pid in candidate_pids]

    ranked.sort(key=lambda item: (-item[1], item[0]))

    return ranked[:topk]

def display_search_results(query, topk=10):
    """Display the ranked top-k search results for a query"""
    ranked_results = search_rank(query, topk)

    if not ranked_results:
        print(f"No results found for query: '{query}'")
        return

    result_pids = [pid for pid, score in ranked_results]

    original_docs = (df.drop_duplicates(subset=["pid"]).set_index("pid", drop=False))

    results = original_docs.loc[result_pids].copy()
    results["score"] = [score for pid, score in ranked_results]

    display_columns = [
        "pid", "title", "description", "brand", "category",
        "sub_category", "product_details", "seller", "out_of_stock",
        "selling_price", "discount", "actual_price",
        "average_rating", "url", "tfidf_score"
    ]
    
    display(results[[col for col in display_columns if col in results.columns]])
    return results

In [10]:
for query in test_queries:
    print("\n" + "=" * 90)
    print("QUERY:", query)
    display_search_results(query, topk=10)



QUERY: women hoodie polycotton red graphic print


,pid,title,description,brand,category,sub_category,product_details,seller,out_of_stock,selling_price,discount,actual_price,average_rating,url
pid,,,,,,,,,,,,,,
SWSFWEF36Z3RKTJ7,SWSFWEF36Z3RKTJ7,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Printed Sweatshir...,Free Authori,Clothing and Accessories,Winter Wear,"[{'Color': 'Red'}, {'Fabric': 'Polycotton'}, {...",BioworldMerchandising,False,849,50% off,"1,699",3.9,https://www.flipkart.com/free-authority-full-s...
SWSFWCSNXA6U57G5,SWSFWCSNXA6U57G5,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Terry S...,Free Authori,Clothing and Accessories,Winter Wear,"[{'Color': 'Red'}, {'Fabric': 'Polycotton'}, {...",BioworldMerchandising,False,799,50% off,"1,599",3.9,https://www.flipkart.com/free-authority-full-s...
SWSFWCSPEGMDH8FV,SWSFWCSPEGMDH8FV,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Sweatsh...,Free Authori,Clothing and Accessories,Winter Wear,"[{'Color': 'Red'}, {'Fabric': 'Polycotton'}, {...",BioworldMerchandising,False,849,50% off,"1,699",3.9,https://www.flipkart.com/free-authority-full-s...
SWSFWCSPHDFH5YKT,SWSFWCSPHDFH5YKT,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Sweatsh...,Free Authori,Clothing and Accessories,Winter Wear,"[{'Color': 'Red'}, {'Fabric': 'Polycotton'}, {...",BioworldMerchandising,False,949,50% off,"1,899",3.9,https://www.flipkart.com/free-authority-full-s...
SWSFWCXHFRX2DBSG,SWSFWCXHFRX2DBSG,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Sweatsh...,Free Authori,Clothing and Accessories,Winter Wear,"[{'Color': 'Red'}, {'Fabric': 'Polycotton'}, {...",BioworldMerchandising,False,799,50% off,"1,599",3.9,https://www.flipkart.com/free-authority-full-s...
SWSFWCXMJRZDWKVU,SWSFWCXMJRZDWKVU,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Sweatsh...,Free Authori,Clothing and Accessories,Winter Wear,"[{'Color': 'Red'}, {'Fabric': 'Polycotton'}, {...",BioworldMerchandising,False,824,45% off,"1,499",3.9,https://www.flipkart.com/free-authority-full-s...
SWSFWEYZC9EEYB6E,SWSFWEYZC9EEYB6E,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Terry S...,Free Authori,Clothing and Accessories,Winter Wear,"[{'Color': 'Red'}, {'Fabric': 'Polycotton'}, {...",BioworldMerchandising,False,824,45% off,"1,499",3.9,https://www.flipkart.com/free-authority-full-s...
SWSFWEYZVYXHPGQJ,SWSFWEYZVYXHPGQJ,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Terry S...,Free Authori,Clothing and Accessories,Winter Wear,"[{'Color': 'Red'}, {'Fabric': 'Polycotton'}, {...",BioworldMerchandising,False,849,50% off,"1,699",3.9,https://www.flipkart.com/free-authority-full-s...
SWSFWEYZZZEJEEJW,SWSFWEYZZZEJEEJW,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Terry S...,Free Authori,Clothing and Accessories,Winter Wear,"[{'Color': 'Red'}, {'Fabric': 'Polycotton'}, {...",BioworldMerchandising,False,849,50% off,"1,699",3.9,https://www.flipkart.com/free-authority-full-s...



QUERY: men wide blue v-neck t-shirt


,pid,title,description,brand,category,sub_category,product_details,seller,out_of_stock,selling_price,discount,actual_price,average_rating,url
pid,,,,,,,,,,,,,,
TSHEXZ8KTZVQJECU,TSHEXZ8KTZVQJECU,"Solid Men V-neck Silver, Blue T-Shirt","Cotton Blend Fabric,V-neck Blue-Milanch T-shir...",HUMBE,Clothing and Accessories,Topwear,"[{'Type': 'V-neck'}, {'Sleeve': 'Half Sleeve'}...",HUMBERT,False,299,40% off,499,2,https://www.flipkart.com/humbert-solid-men-v-n...
TSHFNUHHSKB9Y9HM,TSHFNUHHSKB9Y9HM,Solid Men V Neck Blue T-Shirt,Arbour gives you the new trends t-shirt for me...,ARBO,Clothing and Accessories,Topwear,"[{'Type': 'V Neck'}, {'Sleeve': 'Full Sleeve'}...",ARBOR,False,360,39% off,599,3.9,https://www.flipkart.com/arbour-solid-men-v-ne...
TSHFNUGTDBPCWZ3Q,TSHFNUGTDBPCWZ3Q,Solid Men V Neck Light Blue T-Shirt,Arbour gives you the new trends t-shirt for me...,ARBO,Clothing and Accessories,Topwear,"[{'Type': 'V Neck'}, {'Sleeve': 'Full Sleeve'}...",ARBOR,False,360,39% off,599,3.9,https://www.flipkart.com/arbour-solid-men-v-ne...
TSHFACN2M6QJZG6Z,TSHFACN2M6QJZG6Z,Solid Men V-Neck Blue T-Shirt,,Dudli,Clothing and Accessories,Topwear,"[{'Type': 'V-Neck'}, {'Sleeve': 'Half Sleeve'}...",DudlindFashion,False,299,70% off,999,4.6,https://www.flipkart.com/dudlind-solid-men-v-n...
TSHENWW53TW4EGWG,TSHENWW53TW4EGWG,Solid Men V-neck Multicolor T-Shirt (Pack of 3),"Cotton Blend Fabric,V-neck Charcoal, Royal Blu...",HUMBE,Clothing and Accessories,Topwear,"[{'Type': 'V-neck'}, {'Sleeve': 'Half Sleeve'}...",HUMBERT,False,599,33% off,899,4.3,https://www.flipkart.com/humbert-solid-men-v-n...
TSHEXZ8KJC9GHZCH,TSHEXZ8KJC9GHZCH,Solid Men V-neck Multicolor T-Shirt (Pack of 3),"Cotton Blend Fabric,V-neck Blue Milanch, Mint ...",HUMBE,Clothing and Accessories,Topwear,"[{'Type': 'V-neck'}, {'Sleeve': 'Half Sleeve'}...",HUMBERT,False,457,42% off,799,3.9,https://www.flipkart.com/humbert-solid-men-v-n...
TSHFACNYDBQQYHPZ,TSHFACNYDBQQYHPZ,Striped Men V-neck Blue T-Shirt,,Dudli,Clothing and Accessories,Topwear,"[{'Type': 'V-neck'}, {'Sleeve': 'Half Sleeve'}...",DudlindFashion,False,299,70% off,999,3.8,https://www.flipkart.com/dudlind-striped-men-v...
TSHFGH2XTNZFHENV,TSHFGH2XTNZFHENV,"Printed, Solid Men V Neck Red, Blue, Grey T-Sh...",You wiil love the softness and durabbility of ...,Amp,Clothing and Accessories,Topwear,"[{'Type': 'V Neck'}, {'Sleeve': 'Short Sleeve'...",AFFGARMENTS,False,525,64% off,"1,499",3.4,https://www.flipkart.com/ample-printed-solid-m...
TSHE7GY3D5WWFKZR,TSHE7GY3D5WWFKZR,Printed Men V-neck Light Blue T-Shirt,,T10 Spor,Clothing and Accessories,Topwear,"[{'Type': 'V-neck'}, {'Sleeve': 'Half Sleeve'}...",T10Sports,False,478,52% off,999,4,https://www.flipkart.com/t10-sports-printed-me...



QUERY: summer dress pink


,pid,title,description,brand,category,sub_category,product_details,seller,out_of_stock,selling_price,discount,actual_price,average_rating,url
pid,,,,,,,,,,,,,,
TSHFHBNT248XX2GX,TSHFHBNT248XX2GX,"Solid Women Polo Neck Pink, Black, Light Green...",EXPERIENCE THE AMAZING COMFORT OF A KEOTI POLO...,Keo,Clothing and Accessories,Topwear,"[{'Type': 'Polo Neck'}, {'Sleeve': 'Half Sleev...",Keoti,False,664,63% off,"1,799",3.9,https://www.flipkart.com/keoti-solid-men-polo-...
TSHFHBNT4HAKCDBF,TSHFHBNT4HAKCDBF,"Solid Women Polo Neck Pink, Light Green, Yello...",EXPERIENCE THE AMAZING COMFORT OF A KEOTI POLO...,Keo,Clothing and Accessories,Topwear,"[{'Type': 'Polo Neck'}, {'Sleeve': 'Half Sleev...",Keoti,False,664,63% off,"1,799",3.9,https://www.flipkart.com/keoti-solid-men-polo-...
TSHFHBNT5DVSQSVY,TSHFHBNT5DVSQSVY,"Solid Men Polo Neck Light Blue, Green, Pink T-...",EXPERIENCE THE AMAZING COMFORT OF A KEOTI POLO...,Keo,Clothing and Accessories,Topwear,"[{'Type': 'Polo Neck'}, {'Sleeve': 'Half Sleev...",Keoti,False,664,63% off,"1,799",3.8,https://www.flipkart.com/keoti-solid-men-polo-...
TSHFHBNT7PMKUZWT,TSHFHBNT7PMKUZWT,"Solid Women Polo Neck Light Blue, Pink, Light ...",EXPERIENCE THE AMAZING COMFORT OF A KEOTI POLO...,Keo,Clothing and Accessories,Topwear,"[{'Type': 'Polo Neck'}, {'Sleeve': 'Half Sleev...",Keoti,False,664,63% off,"1,799",3.9,https://www.flipkart.com/keoti-solid-men-polo-...
TSHFHBNT82GDFEV8,TSHFHBNT82GDFEV8,"Solid Men Polo Neck Green, Blue, Pink T-Shirt ...",EXPERIENCE THE AMAZING COMFORT OF A KEOTI POLO...,Keo,Clothing and Accessories,Topwear,"[{'Type': 'Polo Neck'}, {'Sleeve': 'Half Sleev...",Keoti,False,664,63% off,"1,799",3.8,https://www.flipkart.com/keoti-solid-men-polo-...
TSHFHBNT8TVE7ZHZ,TSHFHBNT8TVE7ZHZ,"Solid Women Polo Neck Light Blue, Blue, Pink T...",EXPERIENCE THE AMAZING COMFORT OF A KEOTI POLO...,Keo,Clothing and Accessories,Topwear,"[{'Type': 'Polo Neck'}, {'Sleeve': 'Half Sleev...",Keoti,False,664,63% off,"1,799",3.8,https://www.flipkart.com/keoti-solid-men-polo-...
TSHFHBNT9KUPGYQY,TSHFHBNT9KUPGYQY,"Solid Men Polo Neck Green, Pink, Black T-Shirt...",EXPERIENCE THE AMAZING COMFORT OF A KEOTI POLO...,Keo,Clothing and Accessories,Topwear,"[{'Type': 'Polo Neck'}, {'Sleeve': 'Half Sleev...",Keoti,False,664,63% off,"1,799",3.8,https://www.flipkart.com/keoti-solid-men-polo-...
TSHFHBNTAZD28YCW,TSHFHBNTAZD28YCW,"Solid Women Polo Neck Dark Blue, Green, Pink T...",EXPERIENCE THE AMAZING COMFORT OF A KEOTI POLO...,Keo,Clothing and Accessories,Topwear,"[{'Type': 'Polo Neck'}, {'Sleeve': 'Half Sleev...",Keoti,False,664,63% off,"1,799",3.8,https://www.flipkart.com/keoti-solid-men-polo-...
TSHFHBNTBDDNHHYJ,TSHFHBNTBDDNHHYJ,"Solid Men Polo Neck Green, Pink, Light Green T...",EXPERIENCE THE AMAZING COMFORT OF A KEOTI POLO...,Keo,Clothing and Accessories,Topwear,"[{'Type': 'Polo Neck'}, {'Sleeve': 'Half Sleev...",Keoti,False,664,63% off,"1,799",3.8,https://www.flipkart.com/keoti-solid-men-polo-...



QUERY: yellow running shoes


,pid,title,description,brand,category,sub_category,product_details,seller,out_of_stock,selling_price,discount,actual_price,average_rating,url
pid,,,,,,,,,,,,,,
SHOEC87DCYYZH7MF,SHOEC87DCYYZH7MF,"RS-5006 Running Shoes For Women (Black, Yellow)",Vector X Is One Of The Established Sports Bran...,Vector,Footwear,Men's Footwear,"[{'Color': 'Black, Yellow'}, {'Outer Material'...",SOCCERINTERNATIONAL,False,"1,170",9% off,"1,299",4,https://www.flipkart.com/vector-x-rs-5006-runn...
SHOFTZFQQYKC8HY6,SHOFTZFQQYKC8HY6,Running Shoes For Men (Yellow),,Bone,Footwear,Men's Footwear,"[{'Color': 'Yellow'}, {'Outer Material': 'Synt...",Manojsunda,False,341,65% off,999,4.1,https://www.flipkart.com/bonexy-running-shoes-...



QUERY: raincoat waterproof women


,pid,title,description,brand,category,sub_category,product_details,seller,out_of_stock,selling_price,discount,actual_price,average_rating,url
pid,,,,,,,,,,,,,,
RNCFG4PHHGHSA3YH,RNCFG4PHHGHSA3YH,Solid Men Raincoat,Type: rain suit a coat ; a pant waterproof mat...,Wet Off Ho,Clothing and Accessories,Raincoats,"[{'Sales Package': '1 raincoat'}, {'Fabric Car...",X Studio,False,799,55% off,"1,788",3.8,https://www.flipkart.com/wet-off-hood-solid-me...
RNCFWZVGXMGUQDXW,RNCFWZVGXMGUQDXW,Solid Men Raincoat,Type: rain suit a coat ; a pant waterproof mat...,Wet Off Ho,Clothing and Accessories,Raincoats,[{'Sales Package': '1 Piece rain coats for men...,X Studio,False,766,74% off,"2,999",4.4,https://www.flipkart.com/wet-off-hood-solid-me...
RNCFGGTR5JWV8ES9,RNCFGGTR5JWV8ES9,Solid Men Raincoat,Type: rain suit a coat ; a pant waterproof mat...,Wet Off Ho,Clothing and Accessories,Raincoats,"[{'Sales Package': '1 raincoat'}, {'Fabric Car...",X Studio,False,799,55% off,"1,788",3.8,https://www.flipkart.com/wet-off-hood-solid-me...
RNCFQH2E5FGYYXXF,RNCFQH2E5FGYYXXF,Solid Women Raincoat,Raincoats for women Made from fabric Hipura Ny...,The Dry Ca,Clothing and Accessories,Raincoats,"[{'Fabric Care': 'Cold Water Wash Only'}, {'Sl...",Newera,False,"1,095",69% off,"3,588",3.9,https://www.flipkart.com/dry-cape-solid-men-ra...
RNCFQH2EAAE4SDYE,RNCFQH2EAAE4SDYE,Solid Women Raincoat,Raincoats for women Made from fabric Hipura Ny...,The Dry Ca,Clothing and Accessories,Raincoats,"[{'Fabric Care': 'Cold Water Wash Only'}, {'Sl...",Newera,False,"1,095",69% off,"3,588",3.9,https://www.flipkart.com/dry-cape-solid-men-ra...
RNCFY9RXFPXY3HYT,RNCFY9RXFPXY3HYT,Solid Women Raincoat,Shop from a wide range of Bicolor Raincoats fr...,Wet Off Ho,Clothing and Accessories,Raincoats,"[{'Sales Package': '1 raincoat'}, {'Fabric Car...",X Studio,False,995,74% off,"3,888",4.7,https://www.flipkart.com/wet-off-hood-solid-me...
RNCFYMZ6AKKHRJC4,RNCFYMZ6AKKHRJC4,Solid Men Raincoat,Shop from a wide range of Bicolor Raincoats fr...,The Dry Ca,Clothing and Accessories,Raincoats,"[{'Sales Package': '1 piece raincoat'}, {'Fabr...",Newera,False,999,52% off,"2,088",4.7,https://www.flipkart.com/dry-cape-solid-men-ra...
RNCFXRFSHREUUZCN,RNCFXRFSHREUUZCN,Solid Men Raincoat,,Wet Off Ho,Clothing and Accessories,Raincoats,"[{'Sales Package': '1 Pcs Raincoat'}, {'Fabric...",X Studio,False,995,74% off,"3,888",4,https://www.flipkart.com/wet-off-hood-solid-me...
RNCE7A3XWSDRGNWX,RNCE7A3XWSDRGNWX,Galaxy Reverseable Impenetrable Moisture Barri...,Impenetrable Moisture Barrier To Guarantee Dry...,The Dry Ca,Clothing and Accessories,Raincoats,"[{'Sales Package': '1 Rainsuit'}, {'Fabric Car...",Newera,False,845,52% off,"1,788",3.4,https://www.flipkart.com/dry-cape-galaxy-rever...


In [11]:
def precision_at_k(ranked_pids, relevant_pids, k):
    """Calculate precision at k for a ranked list of PIDs"""
    if k <= 0:
        return -1  # Invalid k value

    top_k_pids = ranked_pids[:k]
    if not relevant_pids:
        return 0.0

    relevant_count = sum(pid in relevant_pids for pid in top_k_pids)
    return relevant_count / k

def recall_at_k(ranked_pids, relevant_pids, k):
    """Calculate recall at k for a ranked list of PIDs"""
    if k <= 0:
        return -1  # Invalid k value

    top_k_pids = ranked_pids[:k]
    if not relevant_pids:
        return 0.0  # No relevant documents to recall

    relevant_count = sum(pid in relevant_pids for pid in top_k_pids)
    return relevant_count / len(relevant_pids)

def average_precision_at_k(ranked_pids, relevant_pids, k):
    """Calculate average precision at k for a ranked list of PIDs"""
    if k <= 0:
        return -1  # Invalid k value

    top_k_pids = ranked_pids[:k]
    if not relevant_pids:
        return 0.0  # No relevant documents to evaluate

    hits = 0
    precision_sum = 0.0

    for rank, pid in enumerate(top_k_pids, start=1):
        if pid in relevant_pids:
            hits += 1
            precision_sum += hits/rank

    denominator = min(len(relevant_pids), k)
    return precision_sum / denominator if denominator > 0 else 0.0

def F1_score_at_k(ranked_pids, relevant_pids, k):
    """Calculate F1 score at k for a ranked list of PIDs"""
    p = precision_at_k(ranked_pids, relevant_pids, k)
    r = recall_at_k(ranked_pids, relevant_pids, k)

    if p + r == 0:
        return 0.0  # Avoid division by zero

    return (2 * p * r)/(p + r)

def mean_average_precision_at_k(ranked_lists, relevance_by_query, k):
    """Calculate mean average precision at k (MAP) for multiple queries"""
    values = [average_precision_at_k(ranked_lists[q], relevance_by_query[q], k) for q in ranked_lists]

    if not values:
        return 0.0  # Avoid division by zero if there are no queries
    
    return float(np.mean(values))

def reciprocal_rank(ranked_pids, relevant_pids):
    """Calculate reciprocal rank for a ranked list of PIDs"""
    for rank, pid in enumerate(ranked_pids, start=1):
        if pid in relevant_pids:
            return 1.0/rank
        
    return 0.0  # No relevant documents found

def mean_reciprocal_rank(ranked_lists, relevance_by_query):
    """Calculate mean reciprocal rank (MRR) for multiple queries"""
    values = [reciprocal_rank(ranked_lists[q], relevance_by_query[q]) for q in ranked_lists]

    return float(np.mean(values)) if values else 0.0

def ndcg_at_k(ranked_pids, relevant_pids, k):
    """Calculate normalized discounted cumulative gain (NDCG) at k for a ranked list of PIDs"""
    if k <= 0:
        return -1  # Invalid k value

    gains = [1 if pid in relevant_pids else 0 for pid in ranked_pids[:k]]

    dcg = sum(gain / math.log2(rank + 1) for rank, gain in enumerate(gains, start=1))

    ideal_relevant = min(len(relevant_pids), k)
    if ideal_relevant == 0:
        return 0.0

    ideal_dcg = sum(1 / math.log2(rank + 1)for rank in range(1, ideal_relevant + 1))
    if ideal_dcg == 0:
        return 0.0

    return dcg / ideal_dcg

### Validation evaluation (K=10)

In [12]:
validation_labels = pd.read_csv("../../data/validation_labels.csv")

validation_queries = {
    1: "women full sleeve sweatshirt cotton",
    2: "men slim jeans blue",
}
evaluation_k = 10

if validation_labels[["pid", "query_id", "labels"]].isna().any().any():
    raise ValueError("Validation labels contain missing required values.")
if set(validation_labels["query_id"].unique()) != set(validation_queries):
    raise ValueError("Validation labels do not contain exactly the predefined queries.")
if not validation_labels["labels"].isin([0, 1]).all():
    raise ValueError("Validation labels must be binary (0 or 1).")
if validation_labels.duplicated(["query_id", "pid"]).any():
    raise ValueError("Validation labels contain duplicate query/PID pairs.")

ranked_pids_by_query = {}
relevant_pids_by_query = {}
for query_id, query in validation_queries.items():
    judgments = validation_labels[validation_labels["query_id"] == query_id]
    ranked_pids_by_query[query_id] = [
        pid for pid, _ in search_rank(query, topk=evaluation_k)
    ]
    relevant_pids_by_query[query_id] = set(
        judgments.loc[judgments["labels"] == 1, "pid"].astype(str)
    )

metric_results = {}
for query_id, ranked_pids in ranked_pids_by_query.items():
    relevant_pids = relevant_pids_by_query[query_id]
    metric_results[f"Query {query_id}"] = {
        "P@10": precision_at_k(ranked_pids, relevant_pids, evaluation_k),
        "R@10": recall_at_k(ranked_pids, relevant_pids, evaluation_k),
        "AP@10": average_precision_at_k(ranked_pids, relevant_pids, evaluation_k),
        "F1@10": F1_score_at_k(ranked_pids, relevant_pids, evaluation_k),
        "NDCG@10": ndcg_at_k(ranked_pids, relevant_pids, evaluation_k),
    }

validation_metrics = pd.DataFrame.from_dict(metric_results, orient="index")
display(validation_metrics.apply(lambda column: column.map(lambda value: f"{value:.3f}")))

aggregate_metrics = pd.DataFrame(
    {
        "MAP@10": [mean_average_precision_at_k(ranked_pids_by_query, relevant_pids_by_query, evaluation_k)],
        "MRR@10": [mean_reciprocal_rank(ranked_pids_by_query, relevant_pids_by_query)],
    },
    index=["Overall"],
)
display(aggregate_metrics.apply(lambda column: column.map(lambda value: f"{value:.3f}")))

,P@10,R@10,AP@10,F1@10,NDCG@10
Query 1,0.100,0.077,0.025,0.087,0.095
Query 2,0.000,0.000,0.000,0.000,0.000


,MAP@10,MRR@10
Overall,0.013,0.125


In [13]:
# Create a complete PID-by-query judgment template.

"""
manual_judgments_path = "manual_test_relevance.csv"

manual_judgments = pd.MultiIndex.from_product(
    [test_queries, index_df["pid"].astype(str).unique()], names=["query", "pid"]).to_frame(index=False)

# Add a relevance column initialized to 0 (not relevant)
manual_judgments["relevance"] = 0

# Save the template to a CSV file if it doesn't already exist
if not os.path.exists(manual_judgments_path):
    manual_judgments.to_csv(manual_judgments_path, index=False)
    print(f"Created {manual_judgments_path}. "
        "Review products and change relevance from 0 to 1 where appropriate."
    )
else:
    print(f"{manual_judgments_path} already exists; it was not overwritten.")

display(manual_judgments.head())
"""


'\nmanual_judgments_path = "manual_test_relevance.csv"\n\nmanual_judgments = pd.MultiIndex.from_product(\n    [test_queries, index_df["pid"].astype(str).unique()], names=["query", "pid"]).to_frame(index=False)\n\n# Add a relevance column initialized to 0 (not relevant)\nmanual_judgments["relevance"] = 0\n\n# Save the template to a CSV file if it doesn\'t already exist\nif not os.path.exists(manual_judgments_path):\n    manual_judgments.to_csv(manual_judgments_path, index=False)\n    print(f"Created {manual_judgments_path}. "\n        "Review products and change relevance from 0 to 1 where appropriate."\n    )\nelse:\n    print(f"{manual_judgments_path} already exists; it was not overwritten.")\n\ndisplay(manual_judgments.head())\n'

In [14]:
# Load the manually assigned relevance judgments
manual_file_path = "manual_test_relevance.csv"
manual_df = pd.read_csv(manual_file_path)

manual_df["pid"] = manual_df["pid"].astype(str)
manual_df["relevance"] = pd.to_numeric(manual_df["relevance"], errors="coerce")

if manual_df[["query", "pid", "relevance"]].isna().any().any():
    raise ValueError("Manual relevance labels contain missing values.")
if not manual_df["relevance"].isin([0, 1]).all():
    raise ValueError("Relevance labels must be binary (0 or 1).")
if manual_df.duplicated(["query", "pid"]).any():
    raise ValueError("Duplicate query/PID pairs found.")

# Store relevant PIDs for each query
relevant_pids_by_query = {}

for query in test_queries:
    judgments = manual_df[manual_df["query"] == query]
    relevant_pids_by_query[query] = set(judgments.loc[judgments["relevance"] == 1, "pid"])

# Retrieve and rank documents for each query
evaluation_k = 10
ranked_pids_by_query = {}

for query in test_queries:
    ranked_results = search_rank(query, topk=evaluation_k)
    ranked_pids_by_query[query] = [str(pid) for pid, score in ranked_results]

# Calculate evaluation metrics
metric_results = {}

for query in test_queries:
    ranked_pids = ranked_pids_by_query[query]
    relevant_pids = relevant_pids_by_query[query]

    metric_results[query] = {
        "P@10": precision_at_k(ranked_pids, relevant_pids, evaluation_k),
        "R@10": recall_at_k(ranked_pids, relevant_pids, evaluation_k),
        "AP@10": average_precision_at_k(ranked_pids, relevant_pids, evaluation_k),
        "F1@10": F1_score_at_k(ranked_pids, relevant_pids, evaluation_k),
        "NDCG@10": ndcg_at_k(ranked_pids, relevant_pids, evaluation_k),
    }


test_metrics = pd.DataFrame.from_dict(metric_results, orient="index")

display(test_metrics.apply(lambda column: column.map(lambda value: f"{value:.3f}")))

# Calculate aggregate metrics
aggregate_metrics = pd.DataFrame(
    {
        "MAP@10": [mean_average_precision_at_k(ranked_pids_by_query, relevant_pids_by_query, evaluation_k)],
        "MRR": [mean_reciprocal_rank(ranked_pids_by_query, relevant_pids_by_query)],
    },
    index=["Overall"],
)

display(aggregate_metrics.apply(lambda column: column.map(lambda value: f"{value:.3f}")))


,P@10,R@10,AP@10,F1@10,NDCG@10
women hoodie polycotton red graphic print,0.000,0.000,0.000,0.000,0.000
men wide blue v-neck t-shirt,0.500,1.000,0.911,0.667,0.971
summer dress pink,0.000,0.000,0.000,0.000,0.000
yellow running shoes,0.200,1.000,1.000,0.333,1.000
raincoat waterproof women,0.500,1.000,0.419,0.667,0.598


,MAP@10,MRR
Overall,0.466,0.450


**Comment on each of the evaluation metrics, stating how they differ, and which information gives each of them. Analyze your results.**

- Precision@10 (P@10): Measures the proportion of the first 10 retrieved documents that are relevant. A high value means that most of the results shown to the user are relevant.

- Recall@10 (R@10): Measures the proportion of all manually judged relevant documents that appear in the first 10 results. It indicates how many relevant products the system manages to retrieve.

- Average Precision@10 (AP@10): Evaluates both relevance and ranking order. It calculates the average precision at each position where a relevant document is retrieved. Higher values indicate that relevant documents appear earlier in the ranking.

- F1@10: Combines precision and recall using their harmonic mean. It provides a balance between retrieving relevant documents and avoiding irrelevant results.

- NDCG@10: Evaluates the quality of the ranking by giving more importance to relevant documents appearing near the top of the results. Unlike precision and recall, it considers the position of each relevant document.

- MAP@10: Mean Average Precision, calculated as the average AP@10 across all queries. It provides an overall measure of ranking quality across the query set.

- MRR: Mean Reciprocal Rank, which measures how early the first relevant document appears in the results. It is calculated by averaging the reciprocal rank of the first relevant result for each query.

The evaluation results show that the search engine performs differently depending on the query. It achieves good results for some queries such as men wide blue v-neck t-shirt, yellow running shoes, and raincoat waterproof women. However, it performs poorly for women hoodie polycotton red graphic print and summer dress pink, where all the evaluation metrics are zero.

The overall MAP@10 of 0.466 and MRR of 0.450 indicate moderate performance across the five queries. However, the individual results reveal important differences:
- "women hoodie polycotton red graphic print": All metrics are 0. The system did not retrieve any manually judged relevant products in the top 10. Upon further inspection, we realize that none of the top product returned by this query are judged as relevant in the manual_test_relevance.csv due to the lack of hood in the sweatshirts.
- "men wide blue v-neck t-shirt": P@10 = 0.500 and R@10 = 1.000. The system retrieved all judged relevant products, although only half of the first 10 results were relevant. Its NDCG@10 of 0.971 indicates a very good ranking order.
- "summer dress pink": All metrics are 0. The system failed to retrieve any manually judged relevant products in the top 10. Similarly to the first query, the top 25 products returned by the search engine for this query were judged non-relevant because they were polos instead of dresses.
- "yellow running shoes": P@10 = 0.200 and R@10 = 1.000. All judged relevant products were retrieved, but only two of the first 10 results were relevant. AP@10 and NDCG@10 are both 1.000, indicating that the relevant results were ranked very highly.
- "raincoat waterproof women": P@10 = 0.500 and R@10 = 1.000. All judged relevant products were retrieved, and half of the first 10 results were relevant. However, AP@10 = 0.419 and NDCG@10 = 0.598 indicate that their ranking positions could be improved.

**Analyze the current search system and identify its main problems or limitations. For each issue you find, propose possible ways to resolve it. Consider aspects such as retrieval accuracy, ranking quality, handling of different field types, query formulation, and indexing strategies.**

Although the search engine obtains good results for some queries, the evaluation reveals several limitations that could affect its retrieval accuracy and ranking quality.

- Strict AND matching: The current retrieval system uses conjunctive (AND) matching, meaning that a document must contain all the query terms to be retrieved. This can be too restrictive, especially for longer queries. A possible improvement could be to implement a fallback OR-based retrieval strategy when AND matching returns too few results.
- No understanding of synonyms and vocabulary variations: The current search system relies on exact term matching after preprocessing. Therefore, it may fail to recognise words with similar meanings or different forms. A possible improvement could be to introduce query expansion using synonyms and related terms.
- Fixed field weights: The search engine uses different weights for product fields when calculating the TF-IDF ranking. However, fixed weights might not represent the importance of each field accurately for every query. A possible improvement could be to adjust the weights after experimenting with them and evaluating their impact using the manually assigned relevance labels.
- Limitations of the evaluation process: The manually created ground truth contains only the top 25 products retrieved by the search engine for each query. Therefore, relevant products outside this candidate set are not considered during evaluation. A possible improvement could be to expand the evaluation dataset by manually judging additional products obtained through alternative retrieval strategies.

# **Part 3:** Ranking and Filtering

In [15]:
def compute_idf(term):
    """Calculate smoothed IDF for a term."""
    df_term = document_frequency.get(term, 0)

    return math.log((N_DOCUMENTS + 1) / (df_term + 1)) + 1


# Precompute document norms
norm_squared = defaultdict(float)

for term in vocabulary:
    weighted_tfs = defaultdict(float)

    for field in SEARCHING_FIELDS:
        postings = inverted_index[field].get(term, {})

        for pid, tf in postings.items():
            weighted_tfs[pid] += ((1 + math.log(tf)) * FIELDS_WEIGHTS[field])

    idf = compute_idf(term)

    for pid, weighted_tf in weighted_tfs.items():
        tfidf = weighted_tf * idf
        norm_squared[pid] += tfidf ** 2

document_norms = {pid: math.sqrt(norm_squared[pid])for pid in index_df["pid"]}

print("Document norms calculated successfully!")


def compute_tfidf_cosine_score(pid, query_terms):
    """Calculate TF-IDF cosine similarity between a query and a document."""
    dot_product = 0.0
    query_squared_weights = 0.0

    for term in query_terms:
        idf = compute_idf(term)

        # Query TF is 1 because duplicate terms are removed
        query_weight = idf
        query_squared_weights += query_weight ** 2

        # Calculate the document's weighted TF for this term
        weighted_tf = 0.0

        for field in SEARCHING_FIELDS:
            tf = inverted_index[field].get(term, {}).get(pid, 0)

            if tf > 0:
                weighted_tf += ((1 + math.log(tf)) * FIELDS_WEIGHTS[field])

        document_weight = weighted_tf * idf

        dot_product += query_weight * document_weight

    query_norm = math.sqrt(query_squared_weights)
    doc_norm = document_norms.get(pid, 0.0)

    if query_norm == 0 or doc_norm == 0:
        return 0.0

    return dot_product / (query_norm * doc_norm)


def search_tfidf_cosine(query, topk=10):
    """Retrieve documents using AND semantics and rank using TF-IDF cosine."""
    query_terms = list(dict.fromkeys(preprocess_text(query)))

    if not query_terms:
        return []

    candidate_pids = conjunctive_search(query)

    if not candidate_pids:
        return []

    ranked = [(pid, compute_tfidf_cosine_score(pid, query_terms)) for pid in candidate_pids]
    ranked.sort(key=lambda item: (-item[1], item[0]))

    return ranked[:topk]


print("TF-IDF + cosine implementation completed.")


Document norms calculated successfully!
TF-IDF + cosine implementation completed.


In [16]:
# BM25 parameters
K1 = 1.5
B = 0.75


# Calculate document lengths for every field
document_lengths = {
    field: {row["pid"]: len(row[column]) if isinstance(row[column], list) else 0 for _, row in index_df.iterrows()}
    for field, column in SEARCHING_FIELDS.items()
}


# Calculate average document length for every field
average_field_lengths = {field: np.mean(list(lengths.values())) if lengths else 0 for field, lengths in document_lengths.items()}


def compute_bm25_score(pid, query_terms):
    """Calculate field-weighted BM25 score for one document."""
    score = 0.0

    for term in query_terms:
        df_term = document_frequency.get(term, 0)

        if df_term == 0:
            continue

        # BM25 inverse document frequency
        idf = math.log(1 + (N_DOCUMENTS - df_term + 0.5) / (df_term + 0.5))

        for field in SEARCHING_FIELDS:
            tf = inverted_index[field].get(term, {}).get(pid, 0)

            if tf == 0:
                continue

            doc_length = document_lengths[field].get(pid, 0)
            avgdl = average_field_lengths[field]

            if avgdl == 0:
                continue

            # Document length normalization
            length_normalization = (K1 * (1 - B + B * doc_length / avgdl))

            # BM25 term contribution
            term_score = (idf * (tf * (K1 + 1)) / (tf + length_normalization))

            # Apply the weight of the corresponding field
            score += term_score * FIELDS_WEIGHTS[field]

    return score


def search_bm25(query, topk=10):
    """Retrieve documents using AND semantics and rank using BM25."""
    query_terms = list(dict.fromkeys(preprocess_text(query)))

    if not query_terms:
        return []

    candidate_pids = conjunctive_search(query)

    if not candidate_pids:
        return []

    ranked = [(pid, compute_bm25_score(pid, query_terms)) for pid in candidate_pids]
    ranked.sort(key=lambda x: (-x[1], x[0]))

    return ranked[:topk]


print("BM25 implementation completed.")


BM25 implementation completed.


In [17]:
# Weights for our custom scoring formula
TEXT_WEIGHT = 0.65
RATING_WEIGHT = 0.15
AVAILABILITY_WEIGHT = 0.10
DISCOUNT_WEIGHT = 0.05
VENDOR_WEIGHT = 0.05


# Product metadata indexed by PID
product_metadata = index_df.set_index("pid")


# Create clean vendor identifier
vendor_data = index_df[["pid", "seller"]].copy()
vendor_data["seller_key"] = (vendor_data["seller"].astype("string").str.strip())

# Treat empty vendor names as missing
vendor_data["seller_key"] = vendor_data["seller_key"].replace("", pd.NA)

# Count unique products per vendor
seller_product_counts = (vendor_data.dropna(subset=["seller_key"]).groupby("seller_key")["pid"].nunique())

# Use the 75th percentile as our reference catalogue size
REFERENCE_VENDOR_SIZE = seller_product_counts.quantile(0.75)

print("Number of vendors:", len(seller_product_counts))
print("Median products per vendor:", seller_product_counts.median())
print("75th percentile:", REFERENCE_VENDOR_SIZE)


def compute_vendor_score(seller):
    """Calculate a normalized vendor score based on catalogue size."""
    # Unknown vendors are not automatically penalized
    if pd.isna(seller) or str(seller).strip() == "":
        return 0.5

    seller = str(seller).strip()

    product_count = seller_product_counts.get(seller, 0)

    if REFERENCE_VENDOR_SIZE <= 0:
        return 1.0

    vendor_score = (math.log1p(product_count) / math.log1p(REFERENCE_VENDOR_SIZE))

    return min(1.0, vendor_score)


print("Vendor scoring initialized successfully.")


def compute_custom_score(pid, query_terms):
    """Calculate custom relevance, quality and vendor score."""
    # Textual relevance score using TF-IDF cosine similarity
    text_score = compute_tfidf_cosine_score(pid, query_terms)

    # Retrieve product metadata
    product = product_metadata.loc[str(pid)]

    rating = product["average_rating_num"]
    discount = product["discount_num"]
    out_of_stock = product["out_of_stock_bool"]
    seller = product["seller"]

    # Normalize rating between 0 and 1
    if pd.notna(rating):
        rating_score = np.clip(rating / 5.0, 0, 1)
    else:
        rating_score = 0.0

    # Availability score
    availability_score = 0.0 if out_of_stock else 1.0

    # Normalize discount between 0 and 1
    if pd.notna(discount):
        discount_score = np.clip(discount / 100.0, 0, 1)
    else:
        discount_score = 0.0

    # Vendor catalogue-size score
    vendor_score = compute_vendor_score(seller)

    # Combine all components
    final_score = (
        TEXT_WEIGHT * text_score
        + RATING_WEIGHT * rating_score
        + AVAILABILITY_WEIGHT * availability_score
        + DISCOUNT_WEIGHT * discount_score
        + VENDOR_WEIGHT * vendor_score
    )

    return final_score


def search_custom(query, topk=10):
    """Retrieve documents using AND semantics and custom ranking."""
    query_terms = list(dict.fromkeys(preprocess_text(query)))

    if not query_terms:
        return []

    candidate_pids = conjunctive_search(query)

    if not candidate_pids:
        return []

    ranked = [(pid, compute_custom_score(pid, query_terms)) for pid in candidate_pids]
    ranked.sort(key=lambda x: (-x[1], x[0]))

    return ranked[:topk]


print("Updated custom ranking implementation completed.")

Number of vendors: 534
Median products per vendor: 6.0
75th percentile: 35.75
Vendor scoring initialized successfully.
Updated custom ranking implementation completed.


In [18]:
def compare_ranking_methods(query, topk=10):
    """Compare different ranking methods for a given query."""
    methods = {
        "TF-IDF + Cosine": search_tfidf_cosine,
        "BM25": search_bm25,
        "Custom Score": search_custom
    }

    for method_name, search_function in methods.items():
        print("\n" + "=" * 80)
        print(f"QUERY: {query}")
        print(f"METHOD: {method_name}")
        print("=" * 80)

        ranked_results = search_function(query, topk)

        if not ranked_results:
            print("No results found.")
            continue

        result_pids = [pid for pid, score in ranked_results]

        results = (df.drop_duplicates(subset=["pid"]).assign(pid=lambda x: x["pid"].astype(str)).set_index("pid").loc[result_pids].copy())
        results["score"] = [score for pid, score in ranked_results]

        display(results[["title", "description", "brand", "average_rating_num", "selling_price_num", "out_of_stock_bool", "score"]])


# Compare all methods using the five Part 2 queries
for query in test_queries:
    compare_ranking_methods(query, topk=10)



QUERY: women hoodie polycotton red graphic print
METHOD: TF-IDF + Cosine


,title,description,brand,average_rating_num,selling_price_num,out_of_stock_bool,score
pid,,,,,,,
SWSFWEF36Z3RKTJ7,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Printed Sweatshir...,Free Authori,3.9,849.0,False,0.302547
SWSFWCXHFRX2DBSG,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Sweatsh...,Free Authori,3.9,799.0,False,0.282533
SWSFWCXMJRZDWKVU,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Sweatsh...,Free Authori,3.9,824.0,False,0.282533
SWSFWCSNPEDVPS7Z,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Terry S...,Free Authori,3.9,849.0,False,0.281071
SWSFWCSZGNU7N4F6,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Terry S...,Free Authori,3.9,799.0,False,0.281071
SWSFWCSPEGMDH8FV,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Sweatsh...,Free Authori,3.9,849.0,False,0.279899
SWSFWCSNPENAZRZM,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Terry S...,Free Authori,3.9,799.0,False,0.269275
SWSFWCTDHRABJFCR,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Terry S...,Free Authori,3.9,899.0,False,0.269275
SWSFWEYZZZEJEEJW,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Terry S...,Free Authori,3.9,849.0,False,0.268553



QUERY: women hoodie polycotton red graphic print
METHOD: BM25


,title,description,brand,average_rating_num,selling_price_num,out_of_stock_bool,score
pid,,,,,,,
SWSFWEF36Z3RKTJ7,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Printed Sweatshir...,Free Authori,3.9,849.0,False,29.737720
SWSFWCXHFRX2DBSG,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Sweatsh...,Free Authori,3.9,799.0,False,28.530148
SWSFWCXMJRZDWKVU,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Sweatsh...,Free Authori,3.9,824.0,False,28.530148
SWSFWCSNPEDVPS7Z,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Terry S...,Free Authori,3.9,849.0,False,28.373888
SWSFWCSZGNU7N4F6,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Terry S...,Free Authori,3.9,799.0,False,28.373888
SWSFWCSPEGMDH8FV,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Sweatsh...,Free Authori,3.9,849.0,False,28.347304
SWSFWEYZZZEJEEJW,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Terry S...,Free Authori,3.9,849.0,False,28.279338
SWSFWEYZC9EEYB6E,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Terry S...,Free Authori,3.9,824.0,False,28.240821
SWSFWEYZVYXHPGQJ,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Terry S...,Free Authori,3.9,849.0,False,28.240821



QUERY: women hoodie polycotton red graphic print
METHOD: Custom Score


,title,description,brand,average_rating_num,selling_price_num,out_of_stock_bool,score
pid,,,,,,,
SWSFWEF36Z3RKTJ7,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Printed Sweatshir...,Free Authori,3.9,849.0,False,0.488656
SWSFWCXHFRX2DBSG,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Sweatsh...,Free Authori,3.9,799.0,False,0.475647
SWSFWCSNPEDVPS7Z,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Terry S...,Free Authori,3.9,849.0,False,0.474696
SWSFWCSZGNU7N4F6,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Terry S...,Free Authori,3.9,799.0,False,0.474696
SWSFWCSPEGMDH8FV,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Sweatsh...,Free Authori,3.9,849.0,False,0.473934
SWSFWCXMJRZDWKVU,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Sweatsh...,Free Authori,3.9,824.0,False,0.473147
SWSFWCSNPENAZRZM,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Terry S...,Free Authori,3.9,799.0,False,0.467029
SWSFWCTDHRABJFCR,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Terry S...,Free Authori,3.9,899.0,False,0.467029
SWSFWEYZZZEJEEJW,Full Sleeve Graphic Print Women Sweatshirt,Free Authority Presents this Crew Neck Terry S...,Free Authori,3.9,849.0,False,0.466559


... (20 outputs left out)


QUERY: raincoat waterproof women
METHOD: BM25


,title,description,brand,average_rating_num,selling_price_num,out_of_stock_bool,score
pid,,,,,,,
RNCFG4PHHGHSA3YH,Solid Men Raincoat,Type: rain suit a coat ; a pant waterproof mat...,Wet Off Ho,3.8,799.0,False,77.923707
RNCFWZVGXMGUQDXW,Solid Men Raincoat,Type: rain suit a coat ; a pant waterproof mat...,Wet Off Ho,4.4,766.0,False,75.835409
RNCFGGTR5JWV8ES9,Solid Men Raincoat,Type: rain suit a coat ; a pant waterproof mat...,Wet Off Ho,3.8,799.0,False,75.557885
RNCFQH2E5FGYYXXF,Solid Women Raincoat,Raincoats for women Made from fabric Hipura Ny...,The Dry Ca,3.9,1095.0,False,64.859127
RNCFQH2EAAE4SDYE,Solid Women Raincoat,Raincoats for women Made from fabric Hipura Ny...,The Dry Ca,3.9,1095.0,False,64.859127
RNCFY9RXFPXY3HYT,Solid Women Raincoat,Shop from a wide range of Bicolor Raincoats fr...,Wet Off Ho,4.7,995.0,False,63.780410
RNCFYMZ6AKKHRJC4,Solid Men Raincoat,Shop from a wide range of Bicolor Raincoats fr...,The Dry Ca,4.7,999.0,False,62.792186
RNCFXRFSHREUUZCN,Solid Men Raincoat,,Wet Off Ho,4.0,995.0,False,56.725628
RNCE7BZ7T98DVFU2,Galaxy Solid Women Raincoat,Impenetrable Moisture Barrier To Guarantee Dry...,The Dry Ca,3.4,866.0,False,55.712416



QUERY: raincoat waterproof women
METHOD: Custom Score


,title,description,brand,average_rating_num,selling_price_num,out_of_stock_bool,score
pid,,,,,,,
RNCFWZVGXMGUQDXW,Solid Men Raincoat,Type: rain suit a coat ; a pant waterproof mat...,Wet Off Ho,4.4,766.0,False,0.709428
RNCFXRFSHREUUZCN,Solid Men Raincoat,,Wet Off Ho,4.0,995.0,False,0.708825
RNCFGGTR5JWV8ES9,Solid Men Raincoat,Type: rain suit a coat ; a pant waterproof mat...,Wet Off Ho,3.8,799.0,False,0.694771
RNCFG4PHHGHSA3YH,Solid Men Raincoat,Type: rain suit a coat ; a pant waterproof mat...,Wet Off Ho,3.8,799.0,False,0.693711
RNCFYMZ6AKKHRJC4,Solid Men Raincoat,Shop from a wide range of Bicolor Raincoats fr...,The Dry Ca,4.7,999.0,False,0.676448
RNCFY9RXFPXY3HYT,Solid Women Raincoat,Shop from a wide range of Bicolor Raincoats fr...,Wet Off Ho,4.7,995.0,False,0.657534
RNCFQH2EAAE4SDYE,Solid Women Raincoat,Raincoats for women Made from fabric Hipura Ny...,The Dry Ca,3.9,1095.0,False,0.652240
RNCFQH2E5FGYYXXF,Solid Women Raincoat,Raincoats for women Made from fabric Hipura Ny...,The Dry Ca,3.9,1095.0,False,0.651887
RNCE7BZ7T98DVFU2,Galaxy Solid Women Raincoat,Impenetrable Moisture Barrier To Guarantee Dry...,The Dry Ca,3.4,866.0,False,0.581489


In [19]:
%pip install gensim

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: C:\Users\fronk\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [20]:
# If gensim is not installed, run: %pip install gensim
from gensim.models import Word2Vec

# Combine all searchable preprocessed fields into a single token list per document
doc_tokens_by_pid = {}
corpus_sentences = []

for _, row in index_df.iterrows():
    pid = row["pid"]
    tokens = []
    for column in SEARCHING_FIELDS.values():
        field_tokens = row[column]
        if isinstance(field_tokens, list):
            tokens.extend(field_tokens)
    doc_tokens_by_pid[pid] = tokens
    corpus_sentences.append(tokens)

# Train Word2Vec on the product corpus
w2v_model = Word2Vec(
    sentences=corpus_sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,          # Skip-gram works well on domain-specific product catalogs
    epochs=10,
    seed=42
)

def get_average_vector(tokens, model):
    """Represent a list of tokens as the average of their Word2Vec vectors."""
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not vectors:
        return np.zeros(model.vector_size, dtype=np.float32)
    return np.mean(vectors, axis=0)

# Precompute averaged vectors for all indexed documents
doc_vectors = {
    pid: get_average_vector(tokens, w2v_model)
    for pid, tokens in doc_tokens_by_pid.items()
}

print(f"Word2Vec trained. Vocabulary size: {len(w2v_model.wv)}")
print(f"Precomputed vectors for {len(doc_vectors)} documents.")

Word2Vec trained. Vocabulary size: 20031
Precomputed vectors for 28080 documents.


In [21]:
def cosine_similarity(vec1, vec2):
    """Compute cosine similarity between two 1D numpy vectors."""
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0.0 or norm2 == 0.0:
        return 0.0
    return float(np.dot(vec1, vec2) / (norm1 * norm2))

def search_word2vec(query, topk=20):
    """Retrieve documents using AND semantics and rank by Word2Vec + cosine similarity."""
    query_terms = preprocess_text(query)
    if not query_terms:
        return []

    candidate_pids = conjunctive_search(query)
    if not candidate_pids:
        return []

    query_vec = get_average_vector(query_terms, w2v_model)
    if np.linalg.norm(query_vec) == 0.0:
        return []

    ranked = [
        (pid, cosine_similarity(query_vec, doc_vectors[pid]))
        for pid in candidate_pids
    ]
    ranked.sort(key=lambda item: (-item[1], item[0]))

    return ranked[:topk]

# Display Top-20 results for each of the 5 queries from Part 2
original_docs = (
    df.drop_duplicates(subset=["pid"])
      .assign(pid=lambda x: x["pid"].astype(str))
      .set_index("pid")
)

for query in test_queries:
    print("\n" + "=" * 80)
    print(f"QUERY: {query}")
    print("METHOD: Word2Vec + Cosine (Top-20)")
    print("=" * 80)

    ranked_results = search_word2vec(query, topk=20)
    if not ranked_results:
        print("No results found.")
        continue

    result_pids = [pid for pid, _ in ranked_results]
    results = original_docs.loc[result_pids].copy()
    results["score"] = [score for _, score in ranked_results]

    display(results[["title", "brand", "average_rating_num", "selling_price_num", "out_of_stock_bool", "score"]])


QUERY: women hoodie polycotton red graphic print
METHOD: Word2Vec + Cosine (Top-20)


,title,brand,average_rating_num,selling_price_num,out_of_stock_bool,score
pid,,,,,,
SWSFWCSNPEDVPS7Z,Full Sleeve Graphic Print Women Sweatshirt,Free Authori,3.9,849.0,False,0.791443
SWSFWCSZGNU7N4F6,Full Sleeve Graphic Print Women Sweatshirt,Free Authori,3.9,799.0,False,0.791412
SWSFWCXHFRX2DBSG,Full Sleeve Graphic Print Women Sweatshirt,Free Authori,3.9,799.0,False,0.779307
SWSFWCXMJRZDWKVU,Full Sleeve Graphic Print Women Sweatshirt,Free Authori,3.9,824.0,False,0.779264
SWSFWCSNPENAZRZM,Full Sleeve Graphic Print Women Sweatshirt,Free Authori,3.9,799.0,False,0.778683
SWSFWCTDHRABJFCR,Full Sleeve Graphic Print Women Sweatshirt,Free Authori,3.9,899.0,False,0.778674
SWSFWCSPEGMDH8FV,Full Sleeve Graphic Print Women Sweatshirt,Free Authori,3.9,849.0,False,0.777701
SWSFWCSNXA6U57G5,Full Sleeve Graphic Print Women Sweatshirt,Free Authori,3.9,799.0,False,0.777622
SWSFWEYZVYXHPGQJ,Full Sleeve Graphic Print Women Sweatshirt,Free Authori,3.9,849.0,False,0.777497



QUERY: men wide blue v-neck t-shirt
METHOD: Word2Vec + Cosine (Top-20)


,title,brand,average_rating_num,selling_price_num,out_of_stock_bool,score
pid,,,,,,
TSHFACN2M6QJZG6Z,Solid Men V-Neck Blue T-Shirt,Dudli,4.6,299.0,False,0.871085
TSHE7GY3D5WWFKZR,Printed Men V-neck Light Blue T-Shirt,T10 Spor,4.0,478.0,False,0.869131
TSHF4ZFXV3VHZ5GU,Striped Men V-neck Multicolor T-Shirt,Dudli,2.7,299.0,False,0.868406
TSHFACNYDBQQYHPZ,Striped Men V-neck Blue T-Shirt,Dudli,3.8,299.0,False,0.866446
TSHEXZ8KJC9GHZCH,Solid Men V-neck Multicolor T-Shirt (Pack of 3),HUMBE,3.9,457.0,False,0.858183
TSHEXZ8KTZVQJECU,"Solid Men V-neck Silver, Blue T-Shirt",HUMBE,2.0,299.0,False,0.855053
TSHFNUGTDBPCWZ3Q,Solid Men V Neck Light Blue T-Shirt,ARBO,3.9,360.0,False,0.829167
TSHE8UHZ6WFVCCNU,Printed Men V-neck Multicolor T-Shirt,T10 Spor,4.0,499.0,False,0.820009
TSHENWW53TW4EGWG,Solid Men V-neck Multicolor T-Shirt (Pack of 3),HUMBE,4.3,599.0,False,0.819310



QUERY: summer dress pink
METHOD: Word2Vec + Cosine (Top-20)


,title,brand,average_rating_num,selling_price_num,out_of_stock_bool,score
pid,,,,,,
TSHFHBNT7PMKUZWT,"Solid Women Polo Neck Light Blue, Pink, Light ...",Keo,3.9,664.0,False,0.703460
TSHFHHRKARCRFR79,"Self Design, Solid Women Polo Neck Pink, Yello...",Keo,4.3,664.0,False,0.701933
TSHFHBNTBDDNHHYJ,"Solid Men Polo Neck Green, Pink, Light Green T...",Keo,3.8,664.0,False,0.701569
TSHFHBNT5DVSQSVY,"Solid Men Polo Neck Light Blue, Green, Pink T-...",Keo,3.8,664.0,False,0.701422
TSHFHBNTNU6SRY3B,"Solid Women Polo Neck Dark Blue, Pink, Light G...",Keo,4.7,664.0,False,0.701153
TSHFHBNTWAZT8TCU,"Solid Women Polo Neck Dark Blue, Light Blue, P...",Keo,4.7,664.0,False,0.701081
TSHFHHRKRSHYB6XG,"Self Design, Solid Men Polo Neck Maroon, Pink,...",Keo,3.7,664.0,False,0.700925
TSHFHBNTFMXFGZYG,"Solid Men Polo Neck Dark Blue, Pink, Light Gre...",Keo,3.9,664.0,False,0.700846
TSHFHBNT4HAKCDBF,"Solid Women Polo Neck Pink, Light Green, Yello...",Keo,3.9,664.0,False,0.700685



QUERY: yellow running shoes
METHOD: Word2Vec + Cosine (Top-20)


,title,brand,average_rating_num,selling_price_num,out_of_stock_bool,score
pid,,,,,,
SHOEC87DCYYZH7MF,"RS-5006 Running Shoes For Women (Black, Yellow)",Vector,4.0,1170.0,False,0.746752
SHOFTZFQQYKC8HY6,Running Shoes For Men (Yellow),Bone,4.1,341.0,False,0.728103



QUERY: raincoat waterproof women
METHOD: Word2Vec + Cosine (Top-20)


,title,brand,average_rating_num,selling_price_num,out_of_stock_bool,score
pid,,,,,,
RNCFG4PHHGHSA3YH,Solid Men Raincoat,Wet Off Ho,3.8,799.0,False,0.824749
RNCFWZVGXMGUQDXW,Solid Men Raincoat,Wet Off Ho,4.4,766.0,False,0.821290
RNCFGGTR5JWV8ES9,Solid Men Raincoat,Wet Off Ho,3.8,799.0,False,0.802990
RNCFQH2EAAE4SDYE,Solid Women Raincoat,The Dry Ca,3.9,1095.0,False,0.789134
RNCFQH2E5FGYYXXF,Solid Women Raincoat,The Dry Ca,3.9,1095.0,False,0.787289
RNCFXRFSHREUUZCN,Solid Men Raincoat,Wet Off Ho,4.0,995.0,False,0.784645
RNCE7BZ7T98DVFU2,Galaxy Solid Women Raincoat,The Dry Ca,3.4,866.0,False,0.768695
RNCE7A3XWSDRGNWX,Galaxy Reverseable Impenetrable Moisture Barri...,The Dry Ca,3.4,845.0,False,0.764954
RNCFYMZ6AKKHRJC4,Solid Men Raincoat,The Dry Ca,4.7,999.0,False,0.702234
